# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "nigeria"
fortificant = "folate"
vehicle = "salt"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

vehicle_name  wealth_quintile
salt          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

vehicle_name  wealth_quintile
salt          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

wealth_quintile  vehicle_name
1                salt            0
2                salt            0
3                salt            0
4                salt            0
5                salt            0
Name: value, dtype: int64

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

vehicle_name  wealth_quintile
salt          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [9]:
from lsff_utils import config_utils

scenarios = config_utils.get_intervention_scenarios(location, vehicle)
scenarios

['intervention']

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

vehicle_name  wealth_quintile  sex     age_start  age_end
salt          1                Female  15         50         0.995
              2                Female  15         50         0.997
              3                Female  15         50         0.995
              4                Female  15         50         0.994
              5                Female  15         50         0.985
Name: value, dtype: float64

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

vehicle_name  wealth_quintile
salt          1                  0.995
              2                  0.997
              3                  0.995
              4                  0.994
              5                  0.985
Name: value, dtype: float64

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

vehicle_name  wealth_quintile  sex   
salt          1                Female    0.9
              2                Female    0.9
              3                Female    0.9
              4                Female    0.9
              5                Female    0.9
              1                Male      0.9
              2                Male      0.9
              3                Male      0.9
              4                Male      0.9
              5                Male      0.9
Name: value, dtype: float64

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

vehicle_name  wealth_quintile  sex   
salt          1                Female    0.873
              2                Female    0.873
              3                Female    0.873
              4                Female    0.873
              5                Female    0.873
              1                Male      0.873
              2                Male      0.873
              3                Male      0.873
              4                Male      0.873
              5                Male      0.873
Name: value, dtype: float64

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

vehicle_name  wealth_quintile
salt          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)